In [1]:
# Run once if not already installed
!pip install pandas numpy scikit-learn

In [2]:
import pandas as pd
import numpy as np
import os
from collections import Counter
from sklearn.model_selection import GroupShuffleSplit

# Paths
GUIDE_POSITIVE_PATH = 'SysEvalOffTarget/files/datasets/include_on_targets/GUIDEseq_positive.csv'
GUIDE_NEGATIVE_PATH = 'SysEvalOffTarget/files/datasets/include_on_targets/GUIDEseq_negative.csv'
CHANGE_POSITIVE_PATH = 'SysEvalOffTarget/files/datasets/include_on_targets/CHANGEseq_positive.csv'
CHANGE_NEGATIVE_PATH = 'SysEvalOffTarget/files/datasets/include_on_targets/CHANGEseq_negative.csv'

## Load and Inspect Raw Datasets

In [3]:
guide_seq = pd.concat([pd.read_csv(GUIDE_POSITIVE_PATH), pd.read_csv(GUIDE_NEGATIVE_PATH)], ignore_index=True)
change_seq = pd.concat([pd.read_csv(CHANGE_POSITIVE_PATH), pd.read_csv(CHANGE_NEGATIVE_PATH)], ignore_index=True)

for name, df in [('GUIDE-seq', guide_seq), ('CHANGE-seq', change_seq)]:
    print(f'--- {name} ---')
    print(f'Shape: {df.shape}')
    print(f'Columns: {list(df.columns)}')
    print(f'Dtypes:\n{df.dtypes}')
    print(f'Missing values:\n{df.isnull().sum()}')
    print()

--- GUIDE-seq ---
Shape: (1478003, 13)
Columns: ['Unnamed: 0', 'chrom', 'chromStart', 'chromEnd', 'name', 'GUIDEseq_reads', 'strand', 'offtarget_sequence', 'genomic_coordinate', 'distance', 'target', 'run', 'label']
Dtypes:
Unnamed: 0              int64
chrom                  object
chromStart              int64
chromEnd              float64
name                   object
GUIDEseq_reads        float64
strand                 object
offtarget_sequence     object
genomic_coordinate     object
distance                int64
target                 object
run                   float64
label                   int64
dtype: object
Missing values:
Unnamed: 0                  0
chrom                       0
chromStart                  0
chromEnd              1476301
name                  1476301
GUIDEseq_reads        1476301
strand                      0
offtarget_sequence          0
genomic_coordinate    1476301
distance                    0
target                      0
run                   1476

In [4]:
guide_seq = guide_seq.drop(columns=['Unnamed: 0'])
change_seq = change_seq.drop(columns=['Unnamed: 0', 'Unnamed: 7', 'chromStart:chromEnd'])

In [5]:
guide_seq.head()

,chrom,chromStart,chromEnd,name,GUIDEseq_reads,strand,offtarget_sequence,genomic_coordinate,distance,target,run,label
0,chr19,55115744,55115767.0,AAVS1_site_1,13557.0,+,GTCACCAATCCTGTCCCTAGTGG,chr19:55115745-55115767:+,0,GTCACCAATCCTGTCCCTAGNGG,1.0,1
1,chrX,1450701,1450724.0,AAVS1_site_1,190.0,+,CTCCCCAACCCCATCCCTAGGGG,chrX:1450702-1450724:+,5,GTCACCAATCCTGTCCCTAGNGG,1.0,1
2,chrX,1452125,1452148.0,AAVS1_site_1,105.0,+,CTCCCCAACCCCATCCCTAGGGG,chrX:1452126-1452148:+,5,GTCACCAATCCTGTCCCTAGNGG,1.0,1
3,chr1,12523844,12523867.0,AAVS1_site_1,2.0,+,CACACTAATCCTGTCCCCAGAGG,chr1:12523845-12523867:+,4,GTCACCAATCCTGTCCCTAGNGG,1.0,1
4,chr19,55115744,55115767.0,AAVS1_site_1,55079.0,+,GTCACCAATCCTGTCCCTAGTGG,chr19:55115745-55115767:+,0,GTCACCAATCCTGTCCCTAGNGG,2.0,1


In [6]:
change_seq.head()

,chrom,chromStart,chromEnd,name,CHANGEseq_reads,strand,offtarget_sequence,distance,target,label
0,chr4,121343302,121343325.0,AAVS1_site_1,540.0,+,ATCACCTATCCTATCCCTAAGGG,4,GTCACCAATCCTGTCCCTAGNGG,1
1,chr1,12523844,12523867.0,AAVS1_site_1,314.0,+,CACACTAATCCTGTCCCCAGAGG,4,GTCACCAATCCTGTCCCTAGNGG,1
2,chr8,66370990,66371013.0,AAVS1_site_1,258.0,-,AGCATAAATCCTGTCCCTAGGAG,5,GTCACCAATCCTGTCCCTAGNGG,1
3,chr9,134937838,134937861.0,AAVS1_site_1,226.0,-,AAAACCAAACCTGTCCCTAAAGG,5,GTCACCAATCCTGTCCCTAGNGG,1
4,chr15,36995651,36995674.0,AAVS1_site_1,130.0,+,TGATCCTATCCTGTCCCTAGAGG,5,GTCACCAATCCTGTCCCTAGNGG,1


In [7]:
GRNA_COL = 'target'
TARGET_COL = 'offtarget_sequence'
GUIDE_LABEL_COL = 'GUIDEseq_reads'
CHANGE_LABEL_COL = 'CHANGEseq_reads'

# Checking sequence lengths before filtering
print('GUIDE-seq target lengths:')
print(guide_seq[GRNA_COL].str.len().value_counts())
print('\nGUIDE-seq offtarget lengths:')
print(guide_seq[TARGET_COL].str.len().value_counts())

print('\nCHANGE-seq target lengths:')
print(change_seq[GRNA_COL].str.len().value_counts())
print('\nCHANGE-seq offtarget lengths:')
print(change_seq[TARGET_COL].str.len().value_counts())

GUIDE-seq target lengths:
target
23    1478003
Name: count, dtype: int64

GUIDE-seq offtarget lengths:
offtarget_sequence
23    1478003
Name: count, dtype: int64

CHANGE-seq target lengths:
target
23    2873627
Name: count, dtype: int64

CHANGE-seq offtarget lengths:
offtarget_sequence
23    2873627
Name: count, dtype: int64


## Parse and Align Sequences

Each entry should have:
- A 20 nucleotide gRNA sequence
- A 23 nucleotide genomic target sequence (20nt + 3nt PAM)

In [8]:
def parse_sequences(df, grna_col, target_col):
    df = df.copy()

    # Drop missing rows
    df = df.dropna(subset=[grna_col, target_col])

    df[grna_col] = df[grna_col].str.upper().str.strip()
    df[target_col] = df[target_col].str.upper().str.strip()

    # Both sequences are 23nt (20nt + 3nt PAM)
    valid = (df[grna_col].str.len() == 23) & (df[target_col].str.len() == 23)
    n_dropped = (~valid).sum()
    if n_dropped > 0:
        print(f'Dropping {n_dropped} rows with unexpected sequence lengths')
    return df[valid].reset_index(drop=True)

guide_seq = parse_sequences(guide_seq, GRNA_COL, TARGET_COL)
change_seq = parse_sequences(change_seq, GRNA_COL, TARGET_COL)

print(f'GUIDE-seq after parsing: {guide_seq.shape}')
print(f'CHANGE-seq after parsing: {change_seq.shape}')

GUIDE-seq after parsing: (1478003, 12)
CHANGE-seq after parsing: (2873627, 10)


## Label On-Target vs Off-Target

In GUIDE-seq and CHANGE-seq, entries with a read count above zero are off-target cleavage sites. On-target entries (gRNA perfectly matching the intended site) are labelled 1; off-target labelled 0.

In [9]:
for name, df in [('GUIDE-seq', guide_seq), ('CHANGE-seq', change_seq)]:
    counts = df['label'].value_counts()
    pct = df['label'].value_counts(normalize=True) * 100
    print(f'{name} label distribution:')
    print(pd.DataFrame({'count': counts, 'percent': pct.round(1)}))
    print()

GUIDE-seq label distribution:
         count  percent
label                  
0      1476301     99.9
1         1702      0.1

CHANGE-seq label distribution:
         count  percent
label                  
0      2806151     97.7
1        67476      2.3



## gRNA-Based Train/Test Split

In [10]:
def grna_split(df, grna_col, test_size=0.2, random_state=2000):
    """Split dataset by gRNA identity to prevent leakage."""
    splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    groups = df[grna_col]
    train_idx, test_idx = next(splitter.split(df, groups=groups))
    return df.iloc[train_idx].reset_index(drop=True), df.iloc[test_idx].reset_index(drop=True)

guide_train, guide_test = grna_split(guide_seq, GRNA_COL)
change_train, change_test = grna_split(change_seq, GRNA_COL)

for name, train, test in [
    ('GUIDE-seq', guide_train, guide_test),
    ('CHANGE-seq', change_train, change_test)
]:
    print(f'{name}: train={len(train)}, test={len(test)}')
    print(f'  Train gRNAs: {train[GRNA_COL].nunique()}, Test gRNAs: {test[GRNA_COL].nunique()}')

GUIDE-seq: train=1229372, test=248631
  Train gRNAs: 46, Test gRNAs: 12
CHANGE-seq: train=2352636, test=520991
  Train gRNAs: 88, Test gRNAs: 22


## One-Hot Encoding

In [11]:
NUCLEOTIDES = 'ACGT'

def encode_onehot(df, grna_col, target_col):
    """
    Vectorised one-hot encoding of gRNA and target sequences.
    Output shape: (n_samples, 184) — 46nt total x 4 nucleotides.
    """
    n = len(df)
    grna_len = len(df[grna_col].iloc[0])
    target_len = len(df[target_col].iloc[0])
    
    def encode_column(sequences, seq_len):
        matrix = np.zeros((n, seq_len, 4), dtype=np.float32)
        for i, nuc in enumerate(NUCLEOTIDES):
            for pos in range(seq_len):
                matrix[:, pos, i] = sequences.str[pos] == nuc
        return matrix.reshape(n, -1)
    
    grna_enc = encode_column(df[grna_col], grna_len)
    target_enc = encode_column(df[target_col], target_len)
    return np.concatenate([grna_enc, target_enc], axis=1)

print('Encoding GUIDE-seq...')
X_guide_train = encode_onehot(guide_train, GRNA_COL, TARGET_COL)
X_guide_test = encode_onehot(guide_test, GRNA_COL, TARGET_COL)

print('Encoding CHANGE-seq...')
X_change_train = encode_onehot(change_train, GRNA_COL, TARGET_COL)
X_change_test = encode_onehot(change_test, GRNA_COL, TARGET_COL)

print(f'GUIDE-seq train shape: {X_guide_train.shape}')
print(f'GUIDE-seq test shape: {X_guide_test.shape}')
print(f'CHANGE-seq train shape: {X_change_train.shape}')
print(f'CHANGE-seq test shape: {X_change_test.shape}')

Encoding GUIDE-seq...
Encoding CHANGE-seq...
GUIDE-seq train shape: (1229372, 184)
GUIDE-seq test shape: (248631, 184)
CHANGE-seq train shape: (2352636, 184)
CHANGE-seq test shape: (520991, 184)


In [12]:
OUTPUT_DIR = 'data/processed/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save one-hot arrays
np.save(OUTPUT_DIR + 'X_guide_train.npy', X_guide_train)
np.save(OUTPUT_DIR + 'X_guide_test.npy', X_guide_test)
np.save(OUTPUT_DIR + 'X_change_train.npy', X_change_train)
np.save(OUTPUT_DIR + 'X_change_test.npy', X_change_test)

# Save labels
np.save(OUTPUT_DIR + 'y_guide_train.npy', guide_train['label'].values)
np.save(OUTPUT_DIR + 'y_guide_test.npy', guide_test['label'].values)
np.save(OUTPUT_DIR + 'y_change_train.npy', change_train['label'].values)
np.save(OUTPUT_DIR + 'y_change_test.npy', change_test['label'].values)

# Save dataframes (for later use with DNABERT)
guide_train.to_csv(OUTPUT_DIR + 'guide_train.csv', index=False)
guide_test.to_csv(OUTPUT_DIR + 'guide_test.csv', index=False)
change_train.to_csv(OUTPUT_DIR + 'change_train.csv', index=False)
change_test.to_csv(OUTPUT_DIR + 'change_test.csv', index=False)

print('All files saved to', OUTPUT_DIR)

All files saved to data/processed/
